# Module 8 - Functions Advanced Level

This notebook covers four advanced function techniques you will use regularly: scope (local vs global), variable arguments (`*args`, `**kwargs`), lambda expressions, and docstrings.

## Table of Contents

- [ ] 1. Scope — Local vs Global
- [ ] 2. `*args` — Variable Positional Arguments
- [ ] 3. `**kwargs` — Variable Keyword Arguments
- [ ] 4. Lambda Functions
- [ ] 5. Docstrings — Documenting Functions
- [ ] Summary

---
## 1. Scope — Local vs Global

### What is scope?

**Scope** determines where a variable can be seen and used. Python has two primary scopes:

| Scope | Where defined | Lifetime |
|---|---|---|
| **Local** | Inside a function | Exists only while the function runs |
| **Global** | Outside all functions (module level) | Exists for the entire program |

### Local scope

In [ ]:
# Local scope: variables defined inside a function
def scan_port(port_number):
    scan_result = "open"  # local variable — only exists inside this function
    print(f"Port {port_number}: {scan_result}")

scan_port(443)

# Trying to access scan_result here would cause an error:
# print(scan_result)  # NameError: name 'scan_result' is not defined

### Global scope

In [ ]:
# Global scope: variables defined outside all functions
max_retries = 3  # global variable — accessible everywhere

def attempt_connection():
    # Can read global variables from inside a function
    print(f"Max retries allowed: {max_retries}")

attempt_connection()
print(f"Max retries (global): {max_retries}")

### The `global` keyword

By default, assigning to a variable inside a function creates a **new local variable**, even if a global with the same name exists. To modify a global variable from inside a function, use the `global` keyword.

In [ ]:
# Without global: creates a local variable
failed_attempts = 0  # global

def increment_attempts_wrong():
    failed_attempts = failed_attempts + 1  # ERROR: local variable referenced before assignment

# This would crash:
# increment_attempts_wrong()

In [ ]:
# With global: modifies the global variable
failed_attempts = 0  # global

def increment_attempts_correct():
    global failed_attempts  # declare intent to modify the global
    failed_attempts += 1
    print(f"Failed attempts: {failed_attempts}")

increment_attempts_correct()
increment_attempts_correct()
print(f"Total: {failed_attempts}")

### ⚠️ Best practice: avoid `global` when possible

Modifying global variables from functions makes code harder to test and debug. A better pattern is to **return** the new value:

```python
def increment_attempts(current):
    return current + 1

failed_attempts = increment_attempts(failed_attempts)
```

This keeps the function **pure** — it does not change external state.

### ✅ Check Your Understanding: Scope

#### Exercise 1 — Predict

Before running, predict the output:

```python
port_count = 10

def scan_ports():
    port_count = 5  # local variable
    print(f"Inside function: {port_count}")

scan_ports()
print(f"Outside function: {port_count}")
```

What does each `print()` output?

In [ ]:
# Exercise 1 — Predict the output, then run to verify
# My prediction:
#

port_count = 10

def scan_ports():
    port_count = 5
    print(f"Inside function: {port_count}")

scan_ports()
print(f"Outside function: {port_count}")

<details>
<summary>Click to reveal SOLUTION</summary>

```python
# SOLUTION
# Inside function: 5   — the local variable shadows the global
# Outside function: 10 — the global variable is unchanged
#
# The assignment inside the function created a NEW local variable.
# The global port_count was never modified.
```

</details>

---
## 2. `*args` — Variable Positional Arguments

### What is `*args`?

`*args` allows a function to accept **any number of positional arguments**. The `*` collects all extra arguments into a **tuple**.

```python
def function_name(*args):
    # args is a tuple containing all positional arguments
```

The name `args` is a convention — you could use any name after the `*`, but `args` is standard.

### Why use `*args`?

When you do not know in advance how many arguments the caller will provide.

In [ ]:
# Function that sums any number of CVSS scores
def sum_cvss_scores(*scores):
    print(f"Received {len(scores)} scores: {scores}")
    total = sum(scores)
    return total

# Call with different numbers of arguments
result1 = sum_cvss_scores(9.8)
print(f"Total: {result1}\n")

result2 = sum_cvss_scores(9.8, 7.5, 6.1)
print(f"Total: {result2}\n")

result3 = sum_cvss_scores(4.0, 5.5, 8.8, 9.8, 7.2)
print(f"Total: {result3}")

### Security example: scan multiple ports

In [ ]:
# Scan any number of ports on a given IP
def scan_ports(ip_address, *ports):
    print(f"Scanning {ip_address} on {len(ports)} port(s)...")
    for port_number in ports:
        print(f"  Port {port_number}: open")

# Call with different numbers of ports
scan_ports("192.168.1.10", 80)
print()
scan_ports("192.168.1.11", 22, 80, 443)
print()
scan_ports("192.168.1.12", 22, 80, 443, 3306, 5432)

### ✅ Check Your Understanding: `*args`

#### Exercise 2 — Write

Define a function `calculate_average(*values)` that:
- Accepts any number of numeric arguments
- Returns their average (sum divided by count)

Call it with `(9.8, 7.5, 8.8)` and print the result.

In [ ]:
# Exercise 2 — Function with *args
# Your code here:


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
def calculate_average(*values):
    total = sum(values)
    count = len(values)
    return total / count

avg = calculate_average(9.8, 7.5, 8.8)
print(avg)  # 8.7
```

</details>

---
## 3. `**kwargs` — Variable Keyword Arguments

### What is `**kwargs`?

`**kwargs` allows a function to accept **any number of keyword arguments**. The `**` collects them into a **dictionary**.

```python
def function_name(**kwargs):
    # kwargs is a dict containing all keyword arguments
```

The name `kwargs` is a convention (short for "keyword arguments") — you could use any name after `**`, but `kwargs` is standard.

### Why use `**kwargs`?

When you want the caller to pass named configuration options without pre-defining every possible parameter.

In [ ]:
# Function that creates a server configuration from any keyword arguments
def create_server_config(**config):
    print("Server configuration:")
    for key, value in config.items():
        print(f"  {key}: {value}")

# Call with different configurations
create_server_config(hostname="web01", ip="192.168.1.10", port=443)
print()
create_server_config(hostname="db01", ip="192.168.1.20", port=3306, ssl_enabled=True)

### Security example: create alert with metadata

In [ ]:
# Create a security alert with flexible metadata fields
def create_alert(severity, message, **metadata):
    print(f"[{severity}] {message}")
    if metadata:
        print("Metadata:")
        for key, value in metadata.items():
            print(f"  {key}: {value}")

# Different alerts with different metadata
create_alert("CRITICAL", "SQL injection detected",
             source_ip="10.0.0.50",
             timestamp="2026-02-18 14:30",
             affected_table="users")

print()

create_alert("WARNING", "Failed login attempt",
             username="admin",
             source_ip="203.0.113.42")

### Combining `*args` and `**kwargs`

You can use both in the same function. The order must be:

```python
def func(required, *args, **kwargs):
    ...
```

In [ ]:
# Flexible function signature: required + optional positional + optional keyword
def log_event(event_type, *details, **metadata):
    print(f"Event: {event_type}")
    
    if details:
        print(f"Details: {details}")
    
    if metadata:
        print("Metadata:")
        for key, value in metadata.items():
            print(f"  {key}: {value}")

# Call with all three types of arguments
log_event("LOGIN",
          "User authenticated",
          "Session started",
          username="alice",
          ip="192.168.1.100",
          timestamp="2026-02-18 15:00")

### ✅ Check Your Understanding: `**kwargs`

#### Exercise 3 — Write

Define a function `build_query_string(**params)` that:
- Accepts any number of keyword arguments
- Returns a query string in the format: `"key1=value1&key2=value2"`

Example: `build_query_string(page=1, limit=10)` → `"page=1&limit=10"`

*Hint: use `"&".join(...)` with a list comprehension or loop*

In [ ]:
# Exercise 3 — Function with **kwargs
# Your code here:


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
def build_query_string(**params):
    pairs = []
    for key, value in params.items():
        pairs.append(f"{key}={value}")
    return "&".join(pairs)

query = build_query_string(page=1, limit=10, sort="date")
print(query)  # page=1&limit=10&sort=date
```

</details>

---
## 4. Lambda Functions

### What is a lambda function?

A **lambda** is a small, anonymous function defined with the `lambda` keyword. It can have any number of parameters but only **one expression**.

```python
lambda parameters: expression
```

The expression is evaluated and returned automatically — no `return` keyword needed.

### Lambda vs regular function

In [ ]:
# Regular function
def is_high_port(port_number):
    return port_number >= 1024

# Lambda equivalent
is_high_port_lambda = lambda port_number: port_number >= 1024

# Both work the same way
print("Regular:", is_high_port(8080))
print("Lambda:", is_high_port_lambda(8080))

### When to use lambda

Lambdas are best for **short, throwaway functions** used as arguments to other functions. The most common use case is as the `key` parameter to `sorted()`.

In [ ]:
# Sorting a list of dicts by a specific key using lambda
vulnerabilities = [
    {"name": "SQLi", "cvss": 9.8},
    {"name": "XSS", "cvss": 6.1},
    {"name": "CSRF", "cvss": 8.8},
    {"name": "RCE", "cvss": 10.0}
]

# Sort by CVSS score, highest first
sorted_vulns = sorted(vulnerabilities, key=lambda v: v["cvss"], reverse=True)

print("Vulnerabilities sorted by severity:")
for vuln in sorted_vulns:
    print(f"  {vuln['name']}: CVSS {vuln['cvss']}")

### Security example: filter ports with lambda

In [ ]:
# Filter a list of ports using lambda
all_ports = [21, 22, 23, 80, 135, 443, 445, 3389, 8080]

# Filter: keep only high ports (>= 1024)
high_ports = list(filter(lambda p: p >= 1024, all_ports))
print("High ports:", high_ports)

# Filter: keep only dangerous well-known ports
dangerous = [21, 23, 135, 445, 3389]  # FTP, Telnet, RPC, SMB, RDP
dangerous_found = list(filter(lambda p: p in dangerous, all_ports))
print("Dangerous ports found:", dangerous_found)

### ⚠️ When NOT to use lambda

Do not use lambda for:
- Multi-line logic (lambdas must be a single expression)
- Complex operations (hard to read)
- Functions you will reuse (define a named function instead)

If you need to add a comment to explain the lambda, it is too complex — use a regular function.

### ✅ Check Your Understanding: Lambda

#### Exercise 4 — Write

Given the list of IPs:
```python
ips = ["192.168.1.10", "10.0.0.5", "192.168.1.11", "10.0.0.8"]
```

Use `filter()` with a **lambda** to create a list `private_192` containing only IPs that start with `"192.168"`.

*Hint: use the `.startswith()` string method*

In [ ]:
# Exercise 4 — Filter with lambda
ips = ["192.168.1.10", "10.0.0.5", "192.168.1.11", "10.0.0.8"]

# Your code here:


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
ips = ["192.168.1.10", "10.0.0.5", "192.168.1.11", "10.0.0.8"]

private_192 = list(filter(lambda ip: ip.startswith("192.168"), ips))
print(private_192)  # ['192.168.1.10', '192.168.1.11']
```

</details>

---
## 5. Docstrings — Documenting Functions

### What is a docstring?

A **docstring** is a string literal that appears as the first statement in a function, describing what the function does.

```python
def function_name(param):
    """
    This is a docstring.
    It explains what the function does.
    """
    ...
```

### Why use docstrings?

- **Documentation**: explains the function's purpose to other developers (and your future self)
- **Help system**: accessible via `help(function_name)` in Python
- **Professional standard**: expected in production code

### Docstring format

In [ ]:
# Function with a simple one-line docstring
def calculate_risk_score(severity, likelihood):
    """Calculate risk score as the product of severity and likelihood."""
    return severity * likelihood

# Access the docstring
print(calculate_risk_score.__doc__)

### Multi-line docstring

For more complex functions, use a multi-line docstring with:
- A one-line summary
- A blank line
- Detailed description, parameters, and return value

In [ ]:
def scan_port_range(ip_address, start_port, end_port, timeout=2):
    """
    Scan a range of ports on a target IP address.
    
    Parameters:
        ip_address (str): The target IP address in dotted-quad notation.
        start_port (int): The first port to scan (inclusive).
        end_port (int): The last port to scan (inclusive).
        timeout (int): Connection timeout in seconds (default: 2).
    
    Returns:
        list: A list of integers representing open ports.
    """
    # Simulated implementation
    open_ports = [p for p in range(start_port, end_port + 1) if p % 2 == 0]
    return open_ports

# Use help() to see the docstring formatted nicely
help(scan_port_range)

### Security example: docstring for input validation

In [ ]:
def validate_cve_id(cve_string):
    """
    Validate that a string matches the CVE ID format.
    
    A valid CVE ID has the format 'CVE-YYYY-NNNN' where:
    - YYYY is a 4-digit year
    - NNNN is a 4+ digit number
    
    Parameters:
        cve_string (str): The string to validate.
    
    Returns:
        bool: True if valid, False otherwise.
    """
    if not cve_string.startswith("CVE-"):
        return False
    if len(cve_string) < 13:  # CVE-YYYY-NNNN minimum length
        return False
    return True

# Test it
print(validate_cve_id("CVE-2024-1234"))  # True
print(validate_cve_id("CVE-2024"))       # False

### ✅ Check Your Understanding: Docstrings

#### Exercise 5 — Write

Define a function `is_privileged_port(port_number)` that:
- Returns `True` if the port is below 1024, `False` otherwise
- Has a docstring explaining what it does

Use `help()` to display the docstring after defining the function.

In [ ]:
# Exercise 5 — Function with a docstring
# Your code here:


<details>
<summary><span style="color:#e74c3c; font-size:1.1em;"><strong>💡 SOLUTION</strong></span></summary>

```python
# SOLUTION
def is_privileged_port(port_number):
    """
    Check if a port is privileged (requires root/admin access).
    
    Parameters:
        port_number (int): The port number to check.
    
    Returns:
        bool: True if port < 1024, False otherwise.
    """
    return port_number < 1024

help(is_privileged_port)
```

</details>

---
## Summary

| Concept | Syntax | Use it when |
|---|---|---|
| Local scope | Variables inside a function | Default — variables only needed during function execution |
| Global scope | Variables outside all functions | Data needed across multiple functions |
| `global` keyword | `global var_name` | You must modify a global variable (use sparingly) |
| `*args` | `def func(*args):` | Accept any number of positional arguments |
| `**kwargs` | `def func(**kwargs):` | Accept any number of keyword arguments |
| Lambda | `lambda x: x * 2` | Short, throwaway function (especially for `sorted()`, `filter()`) |
| Docstring | `"""Describe function"""` | Document what the function does (always) |

### Key rules to remember

- Local variables are **created** when the function runs and **destroyed** when it returns
- Use `global` only when absolutely necessary — prefer returning values
- `*args` → tuple, `**kwargs` → dict
- Lambda functions can only contain **one expression** — no statements, no `return`
- Always write docstrings for non-trivial functions

## Next Steps

Practice everything in **[02_Drills_Functions.ipynb](02_Drills_Functions.ipynb)**!